# SentenceTransformer(model_name)
## paraphrase-multilingual-MiniLM-L12-v2

Sprache: multilingual, ~50 Sprachen inkl. Deutsch

Modellgröße: etwas größer → 12 Transformer-Layer (L12)

Embedding-Dimension: 384

Trainingsziel: paraphrase detection / semantic similarity

Vorteil: funktioniert gut für mehrsprachige Sätze, kann ähnliche Sätze erkennen, auch wenn sie in Deutsch, Englisch, Französisch etc. sind

Nachteil: langsamer als L6, etwas größer, Training auf mehr Sprachen → für English-only Anwendungen evtl. unnötig

In [2]:
# !pip install sentence-transformers  # nur einmal nötig

import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

torch.manual_seed(42)

# -----------------------------
# 1. Vorbereitung
# -----------------------------
model_name = "paraphrase-multilingual-MiniLM-L12-v2"  # funktioniert für Deutsch
embedder = SentenceTransformer(model_name)

# Trainingsdaten
pos_texts = [
    "Das ist super!", "Ich liebe es.", "Einfach fantastisch.", "Sehr gut gemacht.", "Ich bin begeistert.",
    "Wunderbare Arbeit.", "Klasse Leistung.", "Absolut empfehlenswert.", "Ein echtes Highlight.", "Top Qualität.",
    "Sehr hilfreich.", "Ich bin sehr zufrieden.", "Großartig!", "Beste Entscheidung.", "Es macht Spaß.",
    "Perfekt gelaufen.", "Toller Service.", "Sehr freundlich.", "Beeindruckend.", "Gerne wieder.",
    "Alles bestens.", "Einwandfrei.", "Hervorragend.", "Spitzenklasse.", "Einfach nur toll."
]

neg_texts = [
    "Das ist schrecklich.", "Ich hasse es.", "Ganz furchtbar.", "Sehr schlecht.", "Ich bin enttäuscht.",
    "Miese Qualität.", "Nicht zu gebrauchen.", "Verschwendung von Zeit.", "Ein totaler Reinfall.", "Unterirdisch.",
    "Überhaupt nicht hilfreich.", "Ich bin unzufrieden.", "Grauenhaft!", "Fehlkauf.", "Es macht keinen Sinn.",
    "Viel zu teuer.", "Schlechter Service.", "Sehr unfreundlich.", "Enttäuschend.", "Nie wieder.",
    "Alles kaputt.", "Mangelhaft.", "Ungenügend.", "Katastrophe.", "Einfach nur mies."
]

train_texts = pos_texts + neg_texts
labels = torch.tensor([1]*25 + [0]*25)

# -----------------------------
# 2. Trainings-Embeddings
# -----------------------------
train_embeddings = torch.tensor(embedder.encode(train_texts))

# -----------------------------
# 3. Klassifikator (MLP)
# -----------------------------
class TransformerClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

embedding_dim = train_embeddings.shape[1]
hidden_dim = 64
num_classes = 2

classifier = TransformerClassifier(embedding_dim, hidden_dim, num_classes)

# -----------------------------
# 4. Training
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)

classifier.train()
for epoch in range(100):
    optimizer.zero_grad()
    outputs = classifier(train_embeddings)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -----------------------------
# 5. Test
# -----------------------------
tests = [
    "Der Kurs ist nicht schlecht.",
    "Der Service ist okay.",
    "Das Produkt ist teuer.",
    "Ich bin überrascht wie gut das ist.",
    "Der Film war langweilig aber schön gefilmt."
]

classifier.eval()
with torch.no_grad():
    test_embeddings = torch.tensor(embedder.encode(tests))
    outputs = classifier(test_embeddings)
    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)
    
    for text, prob, pred in zip(tests, probs, preds):
        label_str = "Positiv" if pred.item() == 1 else "Negativ"
        print(f"\nText: {text}")
        print(f"Wahrscheinlichkeiten (Negativ, Positiv): {prob.numpy()}")
        print(f"Predicted Class: {label_str}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\wug2si\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wug2si\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Epoch 20, Loss: 0.0025
Epoch 40, Loss: 0.0001
Epoch 60, Loss: 0.0001
Epoch 80, Loss: 0.0000
Epoch 100, Loss: 0.0000

Text: Der Kurs ist nicht schlecht.
Wahrscheinlichkeiten (Negativ, Positiv): [3.2720802e-09 1.0000000e+00]
Predicted Class: Positiv

Text: Der Service ist okay.
Wahrscheinlichkeiten (Negativ, Positiv): [9.995156e-16 1.000000e+00]
Predicted Class: Positiv

Text: Das Produkt ist teuer.
Wahrscheinlichkeiten (Negativ, Positiv): [1.000000e+00 8.106555e-10]
Predicted Class: Negativ

Text: Ich bin überrascht wie gut das ist.
Wahrscheinlichkeiten (Negativ, Positiv): [0.12363243 0.87636757]
Predicted Class: Positiv

Text: Der Film war langweilig aber schön gefilmt.
Wahrscheinlichkeiten (Negativ, Positiv): [1.0000000e+00 3.7535374e-11]
Predicted Class: Negativ
